# 03 - Silver Calendar Rates

## Objetivo

Realizar a transformação e padronização das tabelas `Past Rates e Future Rates`,
provenientes da camada Bronze, preparando os dados para análises
de desempenho dos imóveis e posterior avaliação de potencial
de investimento em imóveis para aluguel de curta duração em
João Pessoa - PB.

## Fonte

Tabela de origem:

`airbnb_joao_pessoa.bronze.future_calendar_rates`,
`airbnb_joao_pessoa.bronze.past_calendar_rates`

## Responsabilidades

Nesta etapa serão realizadas:

- avaliação da qualidade dos dados;
- verificação de duplicidades;
- tratamento de valores nulos;
- padronização de tipos de dados;
- padronização de campos textuais;
- validação dos domínios das variáveis;
- tratamento de valores inconsistentes;
- criação de atributos derivados relevantes para análise;
- organização das colunas para a camada analítica.

## Princípio de transformação

A camada Silver tem como objetivo disponibilizar dados
consistentes, padronizados e confiáveis para as etapas
analíticas posteriores.

As transformações serão aplicadas com base na estrutura e
qualidade observadas na camada Bronze, evitando a remoção
de informações sem justificativa.

Os dados originais permanecem preservados na camada Bronze.

## Tabela de saída

`airbnb_joao_pessoa.silver.past_rates`, `airbnb_joao_pessoa.silver.future_rates`

In [0]:
# Leitura
past_rates = spark.table("airbnb_joao_pessoa.bronze.past_calendar_rates")
future_rates = spark.table("airbnb_joao_pessoa.bronze.future_calendar_rates")

display(past_rates)

In [0]:
display(future_rates)

In [0]:
print("Future Rates")
print(f"Quantidade de registros: {future_rates.count()}")
print(f"Quantidade de colunas: {len(future_rates.columns)}")

print("\nPast Rates")
print(f"Quantidade de registros: {past_rates.count()}")
print(f"Quantidade de colunas: {len(past_rates.columns)}")

In [0]:
from pyspark.sql.functions import sum, when

null_summary = (
    listings
    .select([
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in listings.columns
    ])
)

display(null_summary)

Conforme validação do notebook anterior:

| Variável                  | Tipo     | Estratégia inicial                            |
| ------------------------- | -------- | --------------------------------------------- |
| `description`             | texto    | manter nulo                                   |
| `photo_urls`              | texto    | manter nulo                                   |
| `host_name`               | texto    | manter nulo                                   |
| `guests`                  | numérico | investigar antes de preencher                 |
| `bedrooms`                | numérico | investigar antes de preencher                 |
| `registration`            | boolean  | manter nulo / criar flag                      |
| `instant_book`            | boolean  | manter nulo / criar flag                      |
| `professional_management` | boolean  | manter nulo / criar flag                      |
| `checkin_time`            | texto    | manter nulo                                   |
| `checkout_time`           | texto    | manter nulo                                   |
| `guest_favorite`          | boolean  | manter nulo                                   |
| `exact_location`          | boolean  | manter nulo                                   |
| `cleaning_fee`            | numérico | **não transformar nulo em 0 automaticamente** |
| `extra_guest_fee`         | numérico | **não transformar nulo em 0 automaticamente** |
| `single_fee_structure`    | boolean  | manter nulo                                   |


In [0]:
# Padronizando strings
from pyspark.sql.functions import col, trim
text_columns = [
    "listing_name", "listing_type", "room_type", "host_name", "description", "photo_urls", "cancellation_policy",
    "checkin_time", "checkout_time", "currency"
]

for column_name in text_columns:
    listings = listings.withColumn(
        column_name,
        trim(col(column_name))
    )

In [0]:
# padronizando variáveis categóricas

display(listings.select("listing_type").distinct().orderBy("listing_type"))
display(listings.select("room_type").distinct().orderBy("room_type"))
display(listings.select("cancellation_policy").distinct().orderBy("cancellation_policy"))


In [0]:
# invesitgando os nulos de variáveis importantes para a análise

display(
    listings
    .filter(
        col("guests").isNull() |
        col("bedrooms").isNull() |
        col("beds").isNull() |
        col("baths").isNull()
    )
    .select("listing_id", "listing_name", "guests", "bedrooms", "beds", "baths", "room_type", "listing_type")
)

In [0]:
display(listings.select("guests", "bedrooms", "beds", "baths").summary())

In [0]:
# com relação as taxas

display(
    listings.select("listing_id", "guests", "cleaning_fee", "extra_guest_fee", "single_fee_structure")
    .orderBy("listing_id")
)

In [0]:
display(listings.select("cleaning_fee", "extra_guest_fee").summary())

In [0]:
# Criando indicadores de disponibilidade da informação
from pyspark.sql.functions import when

listings = listings.withColumn("has_description", when(col("description").isNotNull(), True).otherwise(False))
listings = listings.withColumn("has_photos", when(col("photo_urls").isNotNull(), True).otherwise(False))

In [0]:
# Validação de valores numéricos
display(
    listings.filter(
        (col("guests") < 0) |
        (col("bedrooms") < 0) |
        (col("beds") < 0) |
        (col("baths") < 0) | 
        (col("cleaning_fee") < 0) |
        (col("extra_guest_fee") < 0)
    )
)

In [0]:
rating_columns = ["rating_overall", "rating_accuracy", "rating_checkin", "rating_cleanliness", "rating_communication", "rating_location", "rating_value"]

for c in rating_columns:
    display(
        listings.filter(
            (col(c) < 0) | (col(c) > 5)
        ).select("listing_id", c)
    )

In [0]:
display(
    listings.filter(
        (col("ttm_occupancy") < 0) |
        (col("ttm_occupancy") > 1) |
         (col("l90d_occupancy") < 0) |
        (col("l90d_occupancy") > 1)
    )
)